<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.8-conversational-rag/notebooks/exercises-6.8.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.8 — Conversational RAG
Netsetos GenAI Course v2.0 | Module 6

Query reformulation, memory architectures, LangChain/LlamaIndex patterns, production session management


In [ ]:
# pip install langchain langchain-openai langchain-community llama-index redis -q
print('Ready for Conversational RAG')


## Ex 1: Query Reformulation — Resolving Pronouns


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

reformulate_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Given chat history and the latest question which might '
     'reference context in the history, formulate a standalone question. '
     'Replace ALL pronouns with specific entities. Do NOT answer.'),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])

# Simulate
history = [
    HumanMessage(content='Tell me about the Netsetos GenAI course'),
    AIMessage(content='The Netsetos GenAI course covers 14 modules...'),
]
follow_up = 'How much does it cost?'

print('Original:', follow_up)
print('Reformulated: How much does the Netsetos GenAI course cost?')
print('\nWithout reformulation, retriever searches for "it" — useless')


## Ex 2: LangChain — create_history_aware_retriever


In [ ]:
# from langchain.chains import create_history_aware_retriever
# from langchain.chains import create_retrieval_chain
# from langchain.chains.combine_documents import create_stuff_documents_chain

# history_aware_retriever = create_history_aware_retriever(
#     llm, retriever, contextualize_q_prompt
# )
# qa_chain = create_stuff_documents_chain(llm, qa_prompt)
# rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

print('create_history_aware_retriever workflow:')
print('  1. Check if chat_history exists')
print('  2. If empty → pass query directly to retriever (skip LLM)')
print('  3. If history → prompt → LLM → standalone query → retriever')
print('  4. Retrieved docs + history + query → QA chain → answer')
print()
print('This replaces deprecated ConversationalRetrievalChain (v0.1.17)')


## Ex 3: LangGraph with Checkpointing


In [ ]:
# from langgraph.graph import StateGraph, START, END, MessagesState
# from langgraph.checkpoint.postgres import PostgresSaver

# workflow = StateGraph(MessagesState)
# workflow.add_node('reformulate', reformulate_query)
# workflow.add_node('retrieve', retrieve_docs)
# workflow.add_node('generate', generate_answer)
# workflow.add_edge(START, 'reformulate')
# workflow.add_edge('reformulate', 'retrieve')
# workflow.add_edge('retrieve', 'generate')
# workflow.add_edge('generate', END)

# with PostgresSaver.from_conn_string('postgresql://...') as cp:
#     cp.setup()
#     graph = workflow.compile(checkpointer=cp)
#     config = {'configurable': {'thread_id': 'user-123'}}
#     result = graph.invoke({'messages': [...]}, config=config)

print('LangGraph vs RunnableWithMessageHistory:')
print('  LangGraph: saves ENTIRE graph state at every node')
print('  RWMH: saves only message history before/after chain')
print('  LangGraph enables: fault recovery, time travel, HITL')


## Ex 4: LlamaIndex ChatEngine — 5 Modes


In [ ]:
# from llama_index.core.memory import ChatMemoryBuffer
# memory = ChatMemoryBuffer.from_defaults(token_limit=3900)

# chat_engine = index.as_chat_engine(
#     chat_mode='condense_plus_context',
#     memory=memory,
#     system_prompt='You are a helpful assistant.',
# )
# response = chat_engine.chat('What is the return policy?')
# response2 = chat_engine.chat('What about after 30 days?')  # auto-uses history

print('5 ChatEngine modes:')
for mode, desc in [
    ('condense_question', 'Reformulates → queries KB. Cannot answer meta-Qs'),
    ('condense_plus_context', 'Reformulates + retrieves context. RECOMMENDED'),
    ('simple', 'No retrieval, pure LLM chat. Fallback.'),
    ('react', 'Agentic: decides retrieve or answer. Highest latency.'),
    ('context', 'Always retrieves. Simple but wasteful for greetings.'),
]: print(f'  {mode:25s}: {desc}')


## Ex 5: 6 Memory Architectures


In [ ]:
print('Memory Architecture Decision Guide:')
print()
for mem, when, limit in [
    ('Buffer', 'Short conversations (<15 turns)', 'Linear token growth'),
    ('Window (k=5-10)', 'Most conversational RAG', 'Abrupt forgetting at k'),
    ('Summary', 'Long conversations', 'Loses detail, adds latency'),
    ('Hybrid Summary-Buffer', 'BEST GENERAL PURPOSE', 'Complexity'),
    ('Entity', 'CRM/medical (track entities)', 'Entity extraction cost'),
    ('Vector', 'References to far-back context', 'Second retrieval step'),
]: print(f'  {mem:25s} | {when:35s} | Limit: {limit}')

print('\nToken Budget (8K model):')
print('  System prompt:  500 tokens (10-15%)')
print('  Chat history:  1,000-1,500 (12-20%)')
print('  Retrieved docs: 2,500-4,000 (30-50%)')
print('  Response:       1,000-2,000 (15-25%)')
print('  KEY: Stay below 80% of total token limit')


## Ex 6: Production Session Management


In [ ]:
# Redis session management
# from redisvl.extensions.session_manager import StandardSessionManager
# session = StandardSessionManager(name='chat', session_tag='user:42')
# session.add_messages([{'role':'user','content':'Hello'}])
# history = session.get_recent(top_k=5)

print('Production session pattern:')
print('  Hot layer: Redis (sub-ms latency, TTL, token counting)')
print('  Cold storage: PostgreSQL/DynamoDB (persistent)')
print('  On restart: reload from cold → Redis')
print('  Summarize: batch compress every 20 messages')
print()
print('Rate limiting formula:')
print('  Per-user limit = (Org API limit × 0.8) / concurrent_users')
print('  Track BOTH RPM and TPM per session')
print()
print('Streaming pattern:')
print('  1. Accept connection')
print('  2. Stream tokens to client')
print('  3. AFTER stream completes → persist to history')
print('  NEVER persist during streaming')


## Ex 7: Guardrails — 3-Tier Architecture


In [ ]:
import math

def false_positive_rate(n_guards, per_guard_accuracy):
    return 1 - per_guard_accuracy ** n_guards

print('Guardrails False Positive Math:')
for n in [1, 3, 5]:
    for acc in [0.90, 0.95, 0.99]:
        fp = false_positive_rate(n, acc)
        print(f'  {n} guards × {acc:.0%} accuracy = {fp:.1%} FP rate')

print('\n3-Tier Architecture:')
print('  Tier 1 (Rule-based, <10ms): regex PII, keyword blocklists')
print('  Tier 2 (ML classifiers, 20-100ms): BERT toxicity, Presidio NER')
print('  Tier 3 (LLM-based, 500ms-8s): Llama Guard, NeMo Guardrails')
print('\nTools: NeMo (Colang DSL), Guardrails AI (validators), LLM Guard (PII vault)')


## Ex 8: Multi-Turn Evaluation


In [ ]:
print('3 Conversational RAG Failure Modes:')
for mode, desc in [
    ('Context drift', 'Retriever fetches irrelevant docs as topic shifts'),
    ('Redundant retrieval', 'Same chunks across turns, wastes context'),
    ('Cross-turn hallucination', 'Generator mixes info from different turns'),
]: print(f'  {mode:25s}: {desc}')

print('\nDeepEval ConversationalTestCase:')
print('  from deepeval.test_case import ConversationalTestCase, LLMTestCase')
print('  test = ConversationalTestCase(turns=[turn1, turn2, ...])')
print('  Metrics: TurnRelevancy, TurnFaithfulness (sliding window)')
print('\nProduction eval stack:')
print('  Pre-deploy: 50-200 golden Q&A pairs (faithfulness >0.8)')
print('  CI/CD: DeepEval + pytest as blocking gate')
print('  Production: Langfuse/TruLens for ongoing monitoring')


## Ex 9: India Conversational RAG


In [ ]:
print('India Conversational RAG Stack:')
print()
print('Hinglish challenges:')
print('  250M+ code-switching speakers')
print('  3× tokenization overhead for Devanagari')
print('  GPT-4 ~32% lower accuracy on Hindi')
print()
print('Pipeline: fastText LID → NFC → IndicXlit → segment → embed')
print()
print('Models:')
for m, d in [
    ('BGE-m3', 'Best open-source multilingual retrieval'),
    ('Vyakyarth', 'Krutrim AI Labs, 10 Indian languages'),
    ('IndicTrans2', '22 languages, 1.1B params'),
    ('Bhashini', 'Gov India, 300+ models, 22 languages'),
    ('MuRIL', 'Google, 17 Indian languages'),
]: print(f'  {m:15s}: {d}')
print()
print('Regulatory (design in from day one):')
print('  DPDP Act: full compliance May 2027, ₹250 crore penalties')
print('  RBI FREE-AI: board-approved AI policies, HITL')
print('  IRDAI: 6-hour cyber incident reporting')


---
## Recap
Conversational RAG adds three capabilities to standard RAG: query reformulation (coreference resolution, standalone questions), conversation memory (6 architectures from buffer to vector), and history-aware retrieval (deciding when to retrieve). LangChain: deprecated ConversationalRetrievalChain → RunnableWithMessageHistory → LangGraph (recommended, checkpoints entire state). LlamaIndex: 5 ChatEngine modes (CondensePlus recommended), ChatMemoryBuffer with token_limit. Memory: hybrid summary-buffer is best general-purpose. Token budgeting: stay below 80%. Production: Redis hot + PostgreSQL/DynamoDB cold, 3-tier guardrails (rule/ML/LLM, watch FP math), post-stream persistence, multi-user isolation. Evaluation: DeepEval ConversationalTestCase for 3 failure modes (context drift, redundant retrieval, cross-turn hallucination). India: BGE-m3/Vyakyarth embeddings, IndicTrans2, Bhashini APIs, DPDP + RBI FREE-AI + IRDAI compliance from day one.
